# 01 Data Cleaning

**Goal**: turn the raw Kaggle file into a clean, analysis-ready table and record every cleaning decision.


#### Step 1: Import the data in the python and load the raw data 

In [89]:
# Import libraries 
import numpy as np
import pandas as pd

df_raw = pd.read_csv("/Users/aj/Downloads/DataCo/DataCoSupplyChainDataset.csv",encoding="latin-1")
df_raw.head(3)   # look at the first 3 rows to get a feel for the data


# A running log of every cleaning step: what you did, why, and how many rows it affected.
cleaning_log = []

def log_step(step, rows_before, rows_after, note):
    """Record one cleaning step so the README can show exactly what changed."""
    cleaning_log.append({"step": step, "rows_before": rows_before,
                         "rows_after": rows_after, "rows_removed": rows_before - rows_after,
                         "note": note})

#### Step 2: Check the columns you need are all there 

In [90]:
# If Kaggle ever renames a column, or you downloaded the wrong file, this fails loudly now
# instead of giving a confusing error twenty cells later.
REQUIRED = [
    "Order Id", "Order Item Id", "order date (DateOrders)", "shipping date (DateOrders)",
    "Days for shipping (real)", "Days for shipment (scheduled)", "Delivery Status",
    "Late_delivery_risk", "Shipping Mode", "Market", "Order Region", "Order Country",
    "Order City", "Order Status", "Customer Segment", "Category Name", "Department Name",
    "Product Name", "Sales", "Order Item Total", "Order Item Discount",
    "Order Item Quantity", "Order Profit Per Order", "Benefit per order",
]

missing_cols = [col for col in REQUIRED if col not in df_raw.columns]
if missing_cols:
    print("Columns in your file:", list(df_raw.columns))
    raise ValueError(f"These expected columns are missing: {missing_cols}")
print("All", len(REQUIRED), "required columns found.")

All 24 required columns found.


#### Step 3: Data types and missing values

In [91]:
# info() shows each column's type and non-null count.
# Look for: dates stored as 'object' (text) -> they need converting in Step 9.
df_raw.info()

# Count missing values per column and turn them into a percentage of all rows.
missing = (
    df_raw.isna().sum()
          .to_frame("missing_rows")
          .assign(missing_pct=lambda t: (t["missing_rows"] / len(df_raw) * 100).round(2))
          .query("missing_rows > 0")              # keep only columns that have gaps
          .sort_values("missing_rows", ascending=False)
)
missing   # a column that is ~100% empty is useless; a few missing values can usually be left alone

<class 'pandas.DataFrame'>
RangeIndex: 180519 entries, 0 to 180518
Data columns (total 53 columns):
 #   Column                         Non-Null Count   Dtype  
---  ------                         --------------   -----  
 0   Type                           180519 non-null  str    
 1   Days for shipping (real)       180519 non-null  int64  
 2   Days for shipment (scheduled)  180519 non-null  int64  
 3   Benefit per order              180519 non-null  float64
 4   Sales per customer             180519 non-null  float64
 5   Delivery Status                180519 non-null  str    
 6   Late_delivery_risk             180519 non-null  int64  
 7   Category Id                    180519 non-null  int64  
 8   Category Name                  180519 non-null  str    
 9   Customer City                  180519 non-null  str    
 10  Customer Country               180519 non-null  str    
 11  Customer Email                 180519 non-null  str    
 12  Customer Fname                 180519 non

,missing_rows,missing_pct
Product Description,180519,100.00
Order Zipcode,155679,86.24
Customer Lname,8,0.00
Customer Zipcode,3,0.00


#### Step 4: Duplicates

In [92]:
# Exact duplicate rows = the same line copied twice. These should be removed.
exact_dupes = df_raw.duplicated().sum()

# Each order item should appear once. Duplicated Order Item Ids would double-count sales.
item_id_dupes = df_raw["Order Item Id"].duplicated().sum()

print(f"Exact duplicate rows:          {exact_dupes:,}")
print(f"Duplicated Order Item Ids:     {item_id_dupes:,}")

Exact duplicate rows:          0
Duplicated Order Item Ids:     0


#### Step 5: What's inside the key categorical columns

In [93]:
# value_counts(normalize=True) shows each category's share of rows.
# This tells you the possible values (e.g. the four Delivery Status labels) and how balanced they are.
for col in ["Delivery Status", "Shipping Mode", "Order Status", "Market", "Customer Segment", "Type"]:
    if col in df_raw.columns:
        print(f"--- {col} ---")
        print(df_raw[col].value_counts(normalize=True).mul(100).round(1).astype(str) + " %")
        print()

--- Delivery Status ---
Delivery Status
Late delivery        54.8 %
Advance shipping     23.0 %
Shipping on time     17.8 %
Shipping canceled     4.3 %
Name: proportion, dtype: str

--- Shipping Mode ---
Shipping Mode
Standard Class    59.7 %
Second Class      19.5 %
First Class       15.4 %
Same Day           5.4 %
Name: proportion, dtype: str

--- Order Status ---
Order Status
COMPLETE           33.0 %
PENDING_PAYMENT    22.1 %
PROCESSING         12.1 %
PENDING            11.2 %
CLOSED             10.9 %
ON_HOLD             5.4 %
SUSPECTED_FRAUD     2.3 %
CANCELED            2.0 %
PAYMENT_REVIEW      1.0 %
Name: proportion, dtype: str

--- Market ---
Market
LATAM           28.6 %
Europe          27.8 %
Pacific Asia    22.9 %
USCA            14.3 %
Africa           6.4 %
Name: proportion, dtype: str

--- Customer Segment ---
Customer Segment
Consumer       51.8 %
Corporate      30.4 %
Home Office    17.9 %
Name: proportion, dtype: str

--- Type ---
Type
DEBIT       38.4 %
TRANSFER    

#### Step 6: Personal and masked columns

In [94]:
# Customer email and password are masked in this dataset (every value is the same placeholder).
# A column with 1 unique value carries no information, and personal data shouldn't sit in a portfolio repo.
for col in ["Customer Email", "Customer Password", "Customer Fname", "Customer Lname",
            "Customer Street", "Product Image", "Product Description", "Order Zipcode"]:
    if col in df_raw.columns:
        print(f"{col:22s} unique values: {df_raw[col].nunique():>7,}   "
              f"missing: {df_raw[col].isna().mean():6.1%}   example: {df_raw[col].dropna().head(1).tolist()}")

Customer Email         unique values:       1   missing:   0.0%   example: ['XXXXXXXXX']
Customer Password      unique values:       1   missing:   0.0%   example: ['XXXXXXXXX']
Customer Fname         unique values:     782   missing:   0.0%   example: ['Cally']
Customer Lname         unique values:   1,109   missing:   0.0%   example: ['Holloway']
Customer Street        unique values:   7,458   missing:   0.0%   example: ['5365 Noble Nectar Island']
Product Image          unique values:     118   missing:   0.0%   example: ['http://images.acmesports.sports/Smart+watch ']
Product Description    unique values:       0   missing: 100.0%   example: []
Order Zipcode          unique values:     609   missing:  86.2%   example: [99301.0]


#### Step 7: Understand the grain (the most important step)

In [95]:
# One row = one ORDER ITEM, not one order. An order with 3 products has 3 rows.
# If you count rows as orders, every number in the project is wrong.
items_per_order = df_raw.groupby("Order Id").size()
print(f"Rows (order items): {len(df_raw):,}")
print(f"Unique orders:      {df_raw['Order Id'].nunique():,}")
print("\nItems per order:")
print(items_per_order.describe().round(2))

Rows (order items): 180,519
Unique orders:      65,752

Items per order:
count    65752.00
mean         2.75
std          1.48
min          1.00
25%          1.00
50%          3.00
75%          4.00
max          5.00
dtype: float64


In [96]:
# Shipping happens per ORDER, so shipping fields should be identical for every item in an order.
# This checks that assumption: ideally every line below says "varies within 0 orders".
order_level_cols = ["Shipping Mode", "Days for shipping (real)", "Days for shipment (scheduled)",
                    "Delivery Status", "Order Region", "order date (DateOrders)"]
for col in order_level_cols:
    n_varying = (df_raw.groupby("Order Id")[col].nunique() > 1).sum()
    print(f"{col:32s} varies within {n_varying:,} orders")

Shipping Mode                    varies within 0 orders
Days for shipping (real)         varies within 0 orders
Days for shipment (scheduled)    varies within 0 orders
Delivery Status                  varies within 0 orders
Order Region                     varies within 0 orders
order date (DateOrders)          varies within 0 orders


In [97]:
# Is "Order Profit Per Order" really per ORDER (repeated on every item), or per ITEM?
# The answer decides how to total profit later: SUM if it's per item, take one value if it's repeated.
multi_item_orders = items_per_order[items_per_order > 1].index
multi = df_raw[df_raw["Order Id"].isin(multi_item_orders)]
share_varies = (multi.groupby("Order Id")["Order Profit Per Order"].nunique() > 1).mean()

PROFIT_AGG = "sum" if share_varies > 0.5 else "first"
print(f"In {share_varies:.0%} of multi-item orders, profit differs between items.")
print(f"-> Profit is item-level, so aggregate it with: '{PROFIT_AGG}'")

# Bonus check: is it the same as "Benefit per order"? If yes, one of the two columns is redundant.
same = (df_raw["Order Profit Per Order"] == df_raw["Benefit per order"]).mean()
print(f"'Order Profit Per Order' equals 'Benefit per order' in {same:.1%} of rows")

In 100% of multi-item orders, profit differs between items.
-> Profit is item-level, so aggregate it with: 'sum'
'Order Profit Per Order' equals 'Benefit per order' in 100.0% of rows


#### Step 8: Drop columns you don't need, remove duplicates

In [98]:
# Start the cleaned table as a copy, so df_raw stays untouched for comparisons.
DROP_COLS = ["Customer Email", "Customer Password", "Customer Fname", "Customer Lname",
             "Customer Street", "Product Image", "Product Description", "Order Zipcode"]
df = df_raw.drop(columns=[c for c in DROP_COLS if c in df_raw.columns]).copy()

# "Benefit per order" duplicates profit if Step 7 showed they're ~100% equal
if same > 0.99:
    df = df.drop(columns=["Benefit per order"])

rows_before = len(df)
df = df.drop_duplicates()
log_step("Drop personal/empty columns + exact duplicates", rows_before, len(df),
         f"Dropped {len(DROP_COLS)} masked/empty/personal columns; removed exact duplicate rows")
print(f"Columns now: {df.shape[1]}   Rows: {len(df):,}")

Columns now: 44   Rows: 180,519


#### Step 9: Convert date text into real dates

In [99]:
def parse_dates(series):
    """Dates look like '1/31/2018 22:56'. Try that exact format first (fast and strict);
    if any value doesn't match, fall back to pandas' automatic parsing."""
    try:
        return pd.to_datetime(series, format="%m/%d/%Y %H:%M")
    except (ValueError, TypeError):
        return pd.to_datetime(series, errors="coerce")   # unparseable values become NaT

df["order_date"] = parse_dates(df["order date (DateOrders)"])
df["ship_date"] = parse_dates(df["shipping date (DateOrders)"])
df = df.drop(columns=["order date (DateOrders)", "shipping date (DateOrders)"])

# NaT = "Not a Time" (a date that failed to parse). This should be 0.
print("Unparsed order dates:", df["order_date"].isna().sum())
print("Unparsed ship dates: ", df["ship_date"].isna().sum())
print("Order dates run from", df["order_date"].min().date(), "to", df["order_date"].max().date())

# Shipping before the order was placed would be a data error
print("Rows shipped before ordered:", (df["ship_date"] < df["order_date"]).sum())

Unparsed order dates: 0
Unparsed ship dates:  0
Order dates run from 2015-01-01 to 2018-01-31
Rows shipped before ordered: 0


#### Step 10: Create the features the analysis needs

In [100]:
# delay_days: positive = late by that many days, 0 = on time, negative = early
df["delay_days"] = df["Days for shipping (real)"] - df["Days for shipment (scheduled)"]

# is_late: 1 if it arrived after the promised date, else 0
df["is_late"] = (df["delay_days"] > 0).astype(int)

# lane = where it went + how it was shipped. This is the unit you'll score for risk.
df["lane"] = df["Order Region"] + " | " + df["Shipping Mode"]

# Time features for trend charts
df["order_month"] = df["order_date"].dt.to_period("M").astype(str)   # e.g. '2017-03'
df["order_year"] = df["order_date"].dt.year
df["order_weekday"] = df["order_date"].dt.day_name()

# Profitability features (guard against division by zero where sales = 0)
df["profit_margin"] = np.where(df["Sales"] > 0, df["Order Profit Per Order"] / df["Sales"], np.nan)
df["is_loss"] = (df["Order Profit Per Order"] < 0).astype(int)

# Sanity check: your late flag should agree with the dataset's own Late_delivery_risk flag
agreement = (df["is_late"] == df["Late_delivery_risk"]).mean()
print(f"is_late matches Late_delivery_risk in {agreement:.2%} of rows")

# Where they disagree, which Delivery Status is it? (Usually cancelled shipments.)
pd.crosstab(df["Delivery Status"], df["is_late"], margins=True)

is_late matches Late_delivery_risk in 97.55% of rows


is_late,0,1,All
Delivery Status,,,
Advance shipping,41592,0,41592
Late delivery,0,98977,98977
Shipping canceled,3331,4423,7754
Shipping on time,32196,0,32196
All,77119,103400,180519


#### Step 11: Set cancelled shipments aside

In [101]:
# A cancelled shipment never shipped, so it can't be "late" or "on time".
# Leaving it in would distort the late rate, so we move it to its own file instead of deleting it.
is_canceled = df["Delivery Status"].eq("Shipping canceled")
canceled = df[is_canceled].copy()

print("Order Status of cancelled shipments:")
print(canceled["Order Status"].value_counts())

rows_before = len(df)
df = df[~is_canceled].copy()
log_step("Set aside cancelled shipments", rows_before, len(df),
         "Shipping canceled rows excluded from late-rate analysis; saved to dataco_canceled.csv")
print(f"\nRows kept for analysis: {len(df):,}   cancelled set aside: {len(canceled):,}")

Order Status of cancelled shipments:
Order Status
SUSPECTED_FRAUD    4062
CANCELED           3692
Name: count, dtype: int64

Rows kept for analysis: 172,765   cancelled set aside: 7,754


#### Step 12: Sanity checks on the numbers

In [102]:
# Each check prints how many rows break a rule. Investigate anything that isn't 0 (or close to it).
checks = {
    "Negative sales":                       (df["Sales"] < 0).sum(),
    "Quantity <= 0":                         (df["Order Item Quantity"] <= 0).sum(),
    "Negative actual shipping days":         (df["Days for shipping (real)"] < 0).sum(),
    "Negative scheduled shipping days":      (df["Days for shipment (scheduled)"] < 0).sum(),
    # Item total should be sales minus discount (allowing 1 cent of rounding)
    "Item total != sales - discount":        (~np.isclose(df["Order Item Total"],
                                                          df["Sales"] - df["Order Item Discount"],
                                                          atol=0.01)).sum(),
}
for name, n in checks.items():
    print(f"{name:40s} {n:,}")

# Loss-making items are REAL (discounts, costs), not errors - keep them.
print(f"\nLoss-making items: {df['is_loss'].mean():.1%} of rows (kept on purpose)")

Negative sales                           0
Quantity <= 0                            0
Negative actual shipping days            0
Negative scheduled shipping days         0
Item total != sales - discount           0

Loss-making items: 18.7% of rows (kept on purpose)


#### Step 13: Rename columns to snake_case

In [103]:
# SQL Server and Power BI are much easier with names like days_for_shipping_real
# than "Days for shipping (real)" (no spaces, brackets or capitals to quote).
df.columns = (df.columns.str.strip()
                        .str.lower()
                        .str.replace(r"[^a-z0-9]+", "_", regex=True)   # any run of other characters -> _
                        .str.strip("_"))
canceled.columns = df.columns

# Put the most-used columns first so the CSV is easy to scan
front = ["order_id", "order_item_id", "order_date", "ship_date", "market", "order_region",
         "shipping_mode", "lane", "days_for_shipment_scheduled", "days_for_shipping_real",
         "delay_days", "is_late", "delivery_status", "category_name", "sales",
         "order_profit_per_order", "profit_margin"]
df = df[front + [c for c in df.columns if c not in front]]
canceled = canceled[df.columns]
print(list(df.columns))

['order_id', 'order_item_id', 'order_date', 'ship_date', 'market', 'order_region', 'shipping_mode', 'lane', 'days_for_shipment_scheduled', 'days_for_shipping_real', 'delay_days', 'is_late', 'delivery_status', 'category_name', 'sales', 'order_profit_per_order', 'profit_margin', 'type', 'sales_per_customer', 'late_delivery_risk', 'category_id', 'customer_city', 'customer_country', 'customer_id', 'customer_segment', 'customer_state', 'customer_zipcode', 'department_id', 'department_name', 'latitude', 'longitude', 'order_city', 'order_country', 'order_customer_id', 'order_item_cardprod_id', 'order_item_discount', 'order_item_discount_rate', 'order_item_product_price', 'order_item_profit_ratio', 'order_item_quantity', 'order_item_total', 'order_state', 'order_status', 'product_card_id', 'product_category_id', 'product_name', 'product_price', 'product_status', 'order_month', 'order_year', 'order_weekday', 'is_loss']


#### Step 14 Save the outputs

In [104]:
df.to_csv(PROCESSED / "dataco_clean.csv", index=False)
canceled.to_csv(PROCESSED / "dataco_canceled.csv", index=False)

log_df = pd.DataFrame(cleaning_log)
log_df.to_csv(PROCESSED / "cleaning_log.csv", index=False)

# Facts the EDA notebook needs, so you don't have to re-derive them
import json
encoding_used = "utf-8"
notes = {"encoding": encoding_used, "profit_aggregation": PROFIT_AGG,
         "rows_clean": len(df), "orders_clean": int(df["order_id"].nunique()),
         "rows_canceled": len(canceled),
         "date_min": str(df["order_date"].min().date()), "date_max": str(df["order_date"].max().date())}
(PROCESSED / "cleaning_notes.json").write_text(json.dumps(notes, indent=2))

print(json.dumps(notes, indent=2))
log_df

{
  "encoding": "utf-8",
  "profit_aggregation": "sum",
  "rows_clean": 172765,
  "orders_clean": 62897,
  "rows_canceled": 7754,
  "date_min": "2015-01-01",
  "date_max": "2018-01-31"
}


,step,rows_before,rows_after,rows_removed,note
0,Drop personal/empty columns + exact duplicates,180519,180519,0,Dropped 8 masked/empty/personal columns; remov...
1,Set aside cancelled shipments,180519,172765,7754,Shipping canceled rows excluded from late-rate...
